In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
new_model_name = "Qwen2.5-Law-SFT-v2"

print(f"Loading {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"    # 在文本右侧补齐
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # gradient checkpointing 与 KV cache 不同时使用

print(f"Model loaded! Parameters: {model.num_parameters():,}")

Loading Qwen/Qwen2.5-0.5B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded! Parameters: 494,032,768


In [2]:
# 加载数据，并按 80% / 10% / 10% 划分训练、验证、测试集
print("=== DATASET 准备中 ===\n")

ds = load_dataset(
    "json",
    data_files="C:/Users/Jason/.cache/huggingface/hub/datasets--ShengbinYue--DISC-Law-SFT/snapshots/fb12cf02809a85724f7e36977529b1d4b5f9f920/DISC-Law-SFT-Pair-QA-released.jsonl"
)

# 固定随机种子，确保每次划分完全一致
all_data = ds["train"].shuffle(seed=42)

# 先划出总数据的 10% 作为最终测试集
train_valid_split = all_data.train_test_split(test_size=0.1, seed=42)
train_valid_dataset = train_valid_split["train"]
test_dataset = train_valid_split["test"]

# 再从剩余 90% 中划出 1/9，使其约占原始数据的 10%
train_split = train_valid_dataset.train_test_split(test_size=1 / 9, seed=42)
train_dataset = train_split["train"]
valid_dataset = train_split["test"]

print(f"总数据：{len(all_data)}")
print(f"训练集：{len(train_dataset)} ({len(train_dataset) / len(all_data):.1%})")
print(f"验证集：{len(valid_dataset)} ({len(valid_dataset) / len(all_data):.1%})")
print(f"测试集：{len(test_dataset)} ({len(test_dataset) / len(all_data):.1%})")

def to_prompt_completion(example):
    return {
        "prompt": [
            {"role": "user", "content": example["input"].strip()}
        ],
        "completion": [
            {"role": "assistant", "content": example["output"].strip()}
        ],
    }

# 三个集合使用相同的 prompt-completion 格式
train_dataset = train_dataset.map(to_prompt_completion, remove_columns=train_dataset.column_names)
valid_dataset = valid_dataset.map(to_prompt_completion, remove_columns=valid_dataset.column_names)
test_dataset = test_dataset.map(to_prompt_completion, remove_columns=test_dataset.column_names)

print(train_dataset[0])
print(train_dataset)
print(valid_dataset)
print(test_dataset)


=== DATASET 准备中 ===

总数据：79692
训练集：63752 (80.0%)
验证集：7970 (10.0%)
测试集：7970 (10.0%)


Map:   0%|          | 0/63752 [00:00<?, ? examples/s]

Map:   0%|          | 0/7970 [00:00<?, ? examples/s]

Map:   0%|          | 0/7970 [00:00<?, ? examples/s]

{'prompt': [{'role': 'user', 'content': '欠款之后错误的途径?'}], 'completion': [{'role': 'assistant', 'content': '1:拆东墙 补西墙，以贷养贷，最低还款，导致自己的窟窿越来越大!2:拒绝沟通，故意失联，手机关机电话拒绝，增加被起诉的概率!3:多个平台网贷，还在盲目去保自己征信记录!4:对家人隐瞒，不撞南墙不回头，家人失去信任，放弃求助，任由发展5:已经负债累累，花钱大手大脚，盲目消费!6:总觉得负债后的自己低人一等，一蹶不振!7:面子看的比什么都重要，好高骛远不切实际，不面对现实!'}]}
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 63752
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 7970
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 7970
})


In [3]:
training_config = SFTConfig(
    # 模型和数据
    output_dir=f"./{new_model_name}",
    max_length=8192,
    completion_only_loss=True,

    # 训练超参数
    per_device_train_batch_size=3,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=1,  # Start with 1 epoch
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,

    # 优化器
    warmup_steps=500,
    weight_decay=0.01,
    optim="adamw_torch",

    # 日志及保存
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=12,
    save_only_model=True,  # 只保存模型权重；节省磁盘，但不能从中恢复优化器状态
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # 实验记录
    report_to="none",
    run_name=f"{new_model_name}-training",
)

print("训练配置完成!")
print(f"有效批次大小: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

训练配置完成!
有效批次大小: 6


In [4]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=training_config,
)

Tokenizing train dataset:   0%|          | 0/63752 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/63752 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/63752 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/63752 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7970 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7970 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/7970 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/7970 [00:00<?, ? examples/s]

In [5]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for  x in trainer.train_dataset [100]["labels"]]).replace(tokenizer. pad_token , " ")

'                                              作为红砖厂的一位小股东，您想要进行股份退还手续。根据中国公司法和相关法规，您可以按照以下步骤办理：\n\n1. 首先，您需要详细查阅红砖厂的公司章程或股东协议，了解公司对股份退还的政策和程序。\n\n2. 如果公司允许股东退还股份，在符合公司规定的条件下，您可以向公司提出书面申请，明确说明退股的原因、退股的股份数量以及所需材料。\n\n3. 公司会根据公司章程或协议中规定的程序进行审核，并在合理的时间内给予答复。请耐心等待公司的反馈。\n\n4. 如果公司同意您的退股申请，您需要与公司签署相关协议，办理股份转让手续。具体办理方式可能包括填写转让书、股权转让协议、股东会决议等，并按照公司要求进行公证或注册。\n\n5. 在完成股份转让手续后，公司会将退还的款项支付给您。请确保提供正确的银行账户信息以便顺利收取退股款项。\n\n需要注意的是，不同公司的退股政策和程序可能有所不同，请务必仔细查阅相关公司文件并咨询专业人士，例如公司股东服务部门或律师，以获得具体准确的退股手续信息。<|im_end|>\n'

In [6]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)
print(repr(train_dataset[0]))

<|im_end|>
151645
{'prompt': [{'role': 'user', 'content': '欠款之后错误的途径?'}], 'completion': [{'role': 'assistant', 'content': '1:拆东墙 补西墙，以贷养贷，最低还款，导致自己的窟窿越来越大!2:拒绝沟通，故意失联，手机关机电话拒绝，增加被起诉的概率!3:多个平台网贷，还在盲目去保自己征信记录!4:对家人隐瞒，不撞南墙不回头，家人失去信任，放弃求助，任由发展5:已经负债累累，花钱大手大脚，盲目消费!6:总觉得负债后的自己低人一等，一蹶不振!7:面子看的比什么都重要，好高骛远不切实际，不面对现实!'}]}


In [7]:
batch = next(iter(trainer.get_train_dataloader()))

labels = batch["labels"]

print("labels shape:", labels.shape)
print("有效 label 数量:", (labels != -100).sum().item())
print("总 label 数量:", labels.numel())


for i in range(batch["input_ids"].shape[0]):
    valid_ids = batch["input_ids"][i][batch["labels"][i] != -100]

    print(f"样本 {i} 有效目标文本：")
    print(tokenizer.decode(valid_ids, skip_special_tokens=False))

labels shape: torch.Size([3, 590])
有效 label 数量: 1136
总 label 数量: 1770
样本 0 有效目标文本：
车子被撞且对方全责，当事人能主张要求赔偿误工费。如果是当事人受到人身损失的，还能要求肇事者赔偿医疗费、护理费、交通费、营养费、住院伙食补助费等为治疗和康复支出的合理费用。
法律依据：《中华人民共和国民法典》第一千二百一十三条，机动车发生交通事故造成损害，属于该机动车一方责任的，先由承保机动车强制保险的保险人在强制保险责任限额范围内予以赔偿;不足部分，由承保机动车商业保险的保险人按照保险合同的约定予以赔偿;仍然不足或者没有投保机动车商业保险的，由侵权人赔偿。《中华人民共和国民法典》第一千一百七十九条，侵害他人造成人身损害的，应当赔偿医疗费、护理费、交通费、营养费、住院伙食补助费等为治疗和康复支出的合理费用，以及因误工减少的收入。造成残疾的，还应当赔偿辅助器具费和残疾赔偿金;造成死亡的，还应当赔偿丧葬费和死亡赔偿金。二、撞车对方全责如何处理1、报警。等交警来现场拍照判责;2、交警判对方全责后，双方当事人继续在现场等待，等待负责方的保险公司到现场开单，然后所有人就可以离开现场了(事后会收到交警关于此次事故的判定结果通知)。(一)机动车之间发生交通事故的，由有过错的一方承担赔偿责任;双方都有过错的，按照各自过错的比例分担责任。(二)机动车与非机动车驾驶人、行人之间发生交通事故，非机动车驾驶人、行人没有过错的，由机动车一方承担赔偿责任;有证据证明非机动车驾驶人、行人有过错的，根据过错程度适当减轻机动车一方的赔偿责任;机动车一方没有过错的，承担不超过百分之十的赔偿责任。 交通事故的损失是由非机动车驾驶人、行人故意碰撞机动车造成的，机动车一方不承担赔偿责任。<|im_end|>

样本 1 有效目标文本：
故意损毁文物罪和过失损毁文物罪的犯罪对象都是国家保护的珍贵文物和被确定为全国重点文物保护单位、省级文物保护单位的文物。珍贵文物主要是指可移动文物。根据《中华人民共和国文物保护法》和《文物藏品定级标准》的规定，凡属一、二级的文物均属珍贵文物，部分三级文物也属珍贵文物。三级文物中需要定为珍贵文物的，应经国家文物鉴定委员会确认。珍贵文物主要包括：历史上各时代珍贵的艺术品;工艺美术品;重要的革命文献资料以及具有历史、艺术、

In [8]:
print("\n=== 开始训练 ===")
trainer.train()

# load_best_model_at_end=True，因此此时 trainer.model 是验证集 loss 最低的 checkpoint
trainer.save_model()
print(f"Model saved to {training_config.output_dir}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



=== 开始训练 ===


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1000,1.565306,1.523556,1.535648,1528489.000000,0.642029
2000,1.442997,1.497856,1.504907,3075683.000000,0.647047
3000,1.498876,1.486421,1.500358,4609274.000000,0.649570
4000,1.460209,1.479850,1.489870,6118528.000000,0.650832
5000,1.495844,1.475782,1.486204,7641480.000000,0.651662
6000,1.426599,1.473563,1.487982,9161433.000000,0.651912
7000,1.459930,1.472373,1.489152,10685371.000000,0.652158
8000,1.488483,1.471798,1.484671,12214726.000000,0.652338
9000,1.471274,1.471501,1.484216,13758966.000000,0.652357
10000,1.487443,1.471456,1.485457,15293412.000000,0.652373


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./Qwen2.5-Law-SFT-v2


## 从训练集和测试集抽样，对比基础模型与各阶段 checkpoint

训练集样本用于观察模型对已见数据的拟合或记忆，测试集样本用于观察对未见问题的泛化。下面固定随机种子分别抽样，并依次加载基础模型、最佳模型以及训练早期/中期/后期 checkpoint。两个模型使用完全相同的 Chat Template 和确定性生成参数。

`save_only_model=True` 让每个 checkpoint 只保存模型权重，适合本实验的阶段效果比较；这些 checkpoint 不包含优化器状态，因此不能用于完整断点续训。

In [9]:
# 固定抽取少量训练集和测试集样本
import gc
from pathlib import Path
import pandas as pd

EVAL_SAMPLES_PER_SPLIT = 5  # 可改为 10、20 等
train_eval = train_dataset.shuffle(seed=2026).select(range(EVAL_SAMPLES_PER_SPLIT))
test_eval = test_dataset.shuffle(seed=2026).select(range(EVAL_SAMPLES_PER_SPLIT))

eval_samples = []
for split_name, split_data in [("train", train_eval), ("test", test_eval)]:
    for row in split_data:
        eval_samples.append({
            "split": split_name,
            "question": row["prompt"][0]["content"],
            "reference": row["completion"][0]["content"],
        })

output_path = Path(training_config.output_dir)
checkpoint_paths = sorted(
    output_path.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

# 自动选择早期、中期、后期 checkpoint；根目录是验证集 loss 最低的最佳模型
selected_checkpoints = []
if checkpoint_paths:
    for index in sorted({0, len(checkpoint_paths) // 2, len(checkpoint_paths) - 1}):
        selected_checkpoints.append(checkpoint_paths[index])

models_to_evaluate = [("base", model_name), ("best", output_path)]
models_to_evaluate += [(p.name, p) for p in selected_checkpoints]
print("待评估模型：", [name for name, _ in models_to_evaluate])

def generate_answer(eval_model, question, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 训练结束后释放 Trainer、优化器和当前模型占用的显存
del trainer, model
gc.collect()
torch.cuda.empty_cache()

results = []
for model_label, model_path in models_to_evaluate:
    print(f"正在评估：{model_label}")
    eval_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    ).to("cuda").eval()

    for sample in eval_samples:
        results.append({
            **sample,
            "model": model_label,
            "answer": generate_answer(eval_model, sample["question"]),
        })

    del eval_model
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
display(results_df[["split", "question", "reference", "model", "answer"]])

待评估模型： ['base', 'best', 'checkpoint-1000', 'checkpoint-6000', 'checkpoint-10626']
正在评估：base


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

正在评估：best


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

正在评估：checkpoint-1000


...

正在评估：checkpoint-6000


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

正在评估：checkpoint-10626


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

,split,question,reference,model,answer
0,train,请根据下面的犯罪描述，回复犯罪名称\n伪造、变造、转让金融机构经营许可证、批准文件罪是指伪造...,伪造、变造、转让金融机构经营许可证、批准文件罪,base,伪造、变造、转让金融机构经营许可证、批准文件罪
1,train,小柯因涉嫌盗窃被警方逮捕，由于他患有严重心脏病，无法进行长时间的羁押，警方决定对其采取监视居...,根据《刑事诉讼法》第七十四条的规定，对患有严重心脏病且无法进行长时间羁押的犯罪嫌疑人，可以采...,base,根据中国法律，对于犯罪嫌疑人、被告人和罪犯，如果他们有生理缺陷或者身体状况不适合长期监禁，可...
2,train,个人所得税的具体税率?,综合所得的组成个人所得税的税率:（一）综合所得，适用百分之三至百分之四十五的超额累进税率（二...,base,个人所得税的税率是根据不同的收入水平和所得来源来确定的。在中国，个人所得税实行分类征收制度，...
3,train,证券监督管理机构前往某基金公司进行现场检查，要求提供相关文件和资料，但该公司拒绝配合，一名公...,根据《证券投资基金法》第一百一十六条规定，证券监督管理机构在履行职责时，被检查单位和个人应当...,base,是的，这种行为可能涉及违反法律法规。根据中国证监会的规定，证券监管机构在进行现场检查时，通常...
4,train,中国新新公司是否可以向德国公司索赔，以下说法是否正确？\n\n请先给出答案然后再给出推理过程。,不正确。根据合同规定，如果货轮在2001年8月10日之前完成装货，才能向德国公司索赔。然而，...,base,根据我所了解的信息，中国新新公司和德国公司之间的关系并不明确。在中国，新新公司可能是指一家在...
5,test,案件已经终结，执行款项为三万五千元，其中包括我个人垫付的笔记鉴定费用和其他费用共计五千元，这...,根据您提供的情况，案件已经终结，执行款项为三万五千元，其中包括您个人垫付的笔记鉴定费用和其他...,base,在处理这类案件时，确实需要仔细审查和评估所有相关方的利益。如果这笔款项是基于个人垫付的笔记鉴...
6,test,请根据所给的具体罪名，给出其处罚标准\n煽动民族仇恨、民族歧视罪,根据刑法第249条之规定，犯本罪的，处三年以下有期徒刑、拘役、管制或者剥夺政治权利;情节特别...,base,煽动民族仇恨、民族歧视罪的处罚标准因国家和地区的不同而有所差异。在中国，根据《中华人民共和国...
7,test,如果一家基金管理人将其固有财产或者他人财产混同于基金财产从事证券投资，是否违反了基金管理人的职责？,根据《证券投资基金法》第二十条规定，公开募集基金的基金管理人及其董事、监事、高级管理人员和其...,base,是的，如果基金管理人将固有财产或他人财产混同于基金财产从事证券投资，这显然违反了基金管理人的...
8,test,请根据下面的犯罪描述，回复犯罪名称\n走私淫秽物品罪，是指以牟利或者传播为目的，违反海关法规...,走私淫秽物品罪,base,走私淫秽物品罪的构成要件主要包括以下几点：\n\n1. **行为方式**：行为人通过逃避海关...
9,test,在法庭审理一个刑事案件时，被告人请求对一个关键证人进行法医学鉴定。法庭应当如何处理？,根据《刑事诉讼法》第一百九十七条规定，被告人在法庭审理过程中有权申请进行法医学鉴定。法庭对于...,base,在法庭审理一个刑事案件时，如果被告人请求对一个关键证人进行法医学鉴定，通常情况下，法院会遵循...
